In [5]:
%pip install --upgrade
%pip install --upgrade transformers datasets[audio] accelerate

KeyboardInterrupt: 

In [ ]:
import os
os.environ["PATH"] += os.pathsep + r"~/home/movemin/myproject/ffmpeg-2026-11-11-git-23c4540053-full_build"

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
# from datasets import load_dataset

# GPU 사용 가능 여부 확인
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-small"

# Whisper 모델 로드
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
    use_safetensors=True
)
model.to(device)

# 오디오 전처리기 및 토크나이저 로드
processor = AutoProcessor.from_pretrained(model_id)

# 음성 인식 파이프라인 생성
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,

    # 파일을 청크 단위로 나누고 청크 시간을 각각 기록
    return_timestamps = True,

    # 10초 단위로 음성 분할
    chunk_length_s = 10,

    # 청크 경계의 앞뒤 5초 문맥 활용
    stride_length_s = 2,
)

# dataset = load_dataset("distil-whisper/librispeech_long", "clean", split = "validation")
# sample = dataset[0]["audio"]
# 분석할 MP3 파일
sample = "./audio/lsy_audio_2023_58s.mp3"

result = pipe(sample)

# print(result["text"])

print(result)


Loading weights: 100%|██████████| 479/479 [00:00<00:00, 6911.53it/s]
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


{'text': ' 안녕하세요. 이 강의는 GPT API로 체법 만들기라는 내용을 다루는 강입니다. GPT API에 대해서 생소하신 분들도 있을텐데, 우리가 잘 알고 있는 채치 GPT, 채치 GPT를 이용해, 채치 GPT의 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 뭐 이런 강의들이 사실 많이 있습니다. 그래서 뭐 여러가지들이 있는데 좀 이 강의 특징이라고 한다면 GPT로 명확한 미션을 달성하는 제법 프로그램을 만드는 게 사실 쉽지는 않은는데 이걸 어떻게 해서 구현을 하는지 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로는 여러가지가 될 수 있는데 여기서 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서 생성하는 프로그램을 만드는 것을 다루려고 합니다. 그래서 프로그램이 실행되는 모습을 한번 보여 드릴게요 우리가 만들 프로그램은 이런식으로 이제 나타나게 되고', 'chunks': [{'timestamp': (0.0, 6.88), 'text': ' 안녕하세요. 이 강의는 GPT API로 체법 만들기라는 내용을 다루는 강입니다.'}, {'timestamp': (6.88, 27.56), 'text': ' GPT API에 대해서 생소하신 분들도 있을텐데, 우리가 잘 알고 있는 채치 GPT, 채치 GPT를 이용해, 채치 GPT의 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 뭐 이런 강의들이 사실 많이 있습니다. 그래서 뭐 여러가지들이 있는데 좀 이 강의 특징이라고 한다면'}, {'timestamp': (27.56, 35.22), 'text': ' GPT로 명확한 미션을 달성하는 제법 프로그램을 만드는 게 사실 쉽지는 않은는데 이걸 어떻게 해서 구현을 하는지'}, {'timestamp': (35.22, 49.8), 'text': ' 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로는 여러가지가 될 수 있는데 여기서 예제

In [8]:
# chunk를 CVS 파일로 저장
start_end_text = []

for chunk in result["chunks"]:
    start = chunk["timestamp"][0]
    end = chunk["timestamp"][1]
    text = chunk["text"]
    start_end_text.append([start, end, text])

import pandas as pd
df = pd.DataFrame(start_end_text, columns = ["start", "end", "text"])
df.to_csv("lsy_audio_2023_58s.csv", index = False, sep = "|")
display(df)

,start,end,text
0,0.00,6.88,안녕하세요. 이 강의는 GPT API로 체법 만들기라는 내용을 다루는 강입니다.
1,6.88,27.56,"GPT API에 대해서 생소하신 분들도 있을텐데, 우리가 잘 알고 있는 채치 GP..."
2,27.56,35.22,GPT로 명확한 미션을 달성하는 제법 프로그램을 만드는 게 사실 쉽지는 않은는데 ...
3,35.22,49.80,그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로는 여러가지가...
4,49.80,54.34,그래서 프로그램이 실행되는 모습을 한번 보여 드릴게요 우리가 만들
5,54.34,58.16,프로그램은 이런식으로 이제 나타나게 되고
